# MvDEC frozen DEKM_F View1: loss-weight and schedule arms - REUTERS

Controlled diagnostic only (`controlled_diagnostic_not_exact_paper_truth`). Each arm is a one-field change to the frozen DEKM_F seed-42 View1 min-500 protocol; the run seed varies View2 only. Each run's stdout/stderr is kept in `logs/` and a label trajectory sidecar is written per run (it never feeds back into training).

- `l4_only`: only the L4 greedy term (L1, L2, L3 weights 0).
- `l2_l3_0p1`: L2 and L3 weights 0.1.
- `kmeans_refresh50`: K-Means refresh every 50 updates instead of 10.
- `refine_lr_1e4`: Adam learning rate 1e-4 during refinement.

REUTERS seeds 42-44 each (12 runs). Kaggle: GPU on, Internet on, attach dataset `datnguyends/mvdec-v1-true-bottleneck-largest-eigen-s42-input`.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

PINNED_COMMIT = "7acc399cad4fc97b74f62516da297b4e6bde93be"
BRANCH = "claude/mvdec-final-round"
REPO_URL = "https://github.com/vdat-18/mvdec_fpmax.git"
REPO = Path("/kaggle/temp/mvdec_fpmax")
OUTPUT = Path("/kaggle/temp/mvdec-frozen-loss-schedule-arms")
RESULTS = Path("/kaggle/working/mvdec-frozen-loss-schedule-arms")
P = "mvdec_2025_"
SUFFIX = "_min500_release_l2_v1"
ARMS = {
    "l4_only": "frozen_dekm_f_s42_view1_true_bottleneck_l4_only",
    "l2_l3_0p1": "frozen_dekm_f_s42_view1_true_bottleneck_l2_l3_0p1",
    "kmeans_refresh50": "frozen_dekm_f_s42_view1_true_bottleneck_kmeans_refresh50",
    "refine_lr_1e4": "frozen_dekm_f_s42_view1_true_bottleneck_refine_lr_1e4",
}
RUNS = [
    {"group": group, "protocol": P + stem + SUFFIX, "dataset": "REUTERS", "seed": seed, "frozen": True}
    for group, stem in ARMS.items()
    for seed in (42, 43, 44)
]


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


EXPECTED_CHECKPOINT_MANIFEST_SHA256 = "71e3c2d423d70bbde1b7423c91c176c1d33a4fb473e7cb6febec187fff8f0607"
EXPECTED_CHECKPOINT_WEIGHT_SHA256 = "ca55a1e995697c346d99540ddd9cc8bd8e6182a7d21d0b09410d7e76b6050702"
candidates = sorted(
    manifest.parent
    for manifest in Path("/kaggle/input").rglob("manifest.json")
    if (manifest.parent / "final.weights.h5").is_file()
    and file_sha256(manifest) == EXPECTED_CHECKPOINT_MANIFEST_SHA256
)
assert len(candidates) == 1, candidates
CHECKPOINT_MANIFEST = candidates[0] / "manifest.json"
assert file_sha256(candidates[0] / "final.weights.h5") == EXPECTED_CHECKPOINT_WEIGHT_SHA256
print("DEKM_F seed-42 View1 checkpoint:", candidates[0])
for run in RUNS:
    if run.pop("frozen", False):
        run["extra"] = ["--view1-manifest", str(CHECKPOINT_MANIFEST)]

for path in (REPO, OUTPUT, RESULTS):
    if path.exists():
        shutil.rmtree(path)
OUTPUT.mkdir(parents=True)
(RESULTS / "trajectory").mkdir(parents=True)
(RESULTS / "logs").mkdir()
subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
TRAINING_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert TRAINING_COMMIT == PINNED_COMMIT, (TRAINING_COMMIT, PINNED_COMMIT)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Training commit:", TRAINING_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"],
    cwd=REPO,
    check=True,
)
if any(run["dataset"] == "20NEWS" for run in RUNS):
    # The official DEKM 20NEWS path reads the scikit-learn cache without downloading.
    from sklearn.datasets import fetch_20newsgroups

    print("20NEWS documents:", len(fetch_20newsgroups(subset="all").data))

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_mvdec_independent_view2.py",
        "tests/test_mvdec_contract.py",
        "tests/test_data.py",
        "tests/test_mvdec_architecture.py",
        "tests/test_mvdec_trajectory.py",
        "tests/test_make_reuters_resample.py",
        "tests/test_mvdec_view2_latent_probe.py",
    ],
    cwd=REPO,
    env={**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO), str(REPO / "src")])},
    check=True,
)

In [ ]:
for run in RUNS:
    tag = f"{run['group']}_{run['dataset']}_seed_{run['seed']}"
    command = [
        sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", run["dataset"],
        "--runs", "1", "--seed", str(run["seed"]),
        "--protocol", run["protocol"],
        *run.get("extra", []),
        "--progress-interval", "100",
        "--dataset-root", run.get("dataset_root", "external_repos/DEKM/datasets"),
        "--public-output-dir", str(OUTPUT / run["group"]),
    ]
    print("Running:", " ".join(command), flush=True)
    env = {**os.environ, "MVDEC_LABEL_TRAJECTORY_LOG": str(RESULTS / "trajectory" / f"{tag}.jsonl")}
    with (RESULTS / "logs" / f"{tag}.log").open("w", encoding="utf-8") as log:
        result = subprocess.run(
            command, cwd=REPO, env=env, check=False, timeout=4 * 60 * 60,
            stdout=log, stderr=subprocess.STDOUT,
        )
    print(tag, "exit code", result.returncode, flush=True)

In [ ]:
import statistics

PAPER = {
    "REUTERS": {"view1": (0.7642, 0.5999), "view2": (0.6501, 0.5129), "fused": (0.7813, 0.6141)},
    "20NEWS": {"view1": (0.4120, 0.4376), "view2": (0.4086, 0.4032), "fused": (0.4163, 0.4543)},
}
rows = []
for path in sorted(OUTPUT.rglob("manifest.json")):
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("status") != "complete":
        continue
    group = path.relative_to(OUTPUT).parts[0]
    rows.append((group, manifest["dataset"], manifest["seed"], manifest["metrics"]))
    target = RESULTS / "runs" / path.parent.relative_to(OUTPUT)
    target.mkdir(parents=True, exist_ok=True)
    for small in path.parent.iterdir():
        if small.is_file() and small.suffix in {".json", ".csv", ".txt"}:
            shutil.copy2(small, target / small.name)
rows.sort(key=lambda row: (row[0], row[1], row[2]))
print("complete runs:", len(rows), "of", len(RUNS))
for key in sorted({(group, dataset) for group, dataset, _, _ in rows}):
    picked = [(seed, metrics) for group, dataset, seed, metrics in rows if (group, dataset) == key]
    print()
    print(*key)
    for seed, metrics in picked:
        print(seed, "  ".join("%s %.4f/%.4f" % (k, metrics[k + "_acc"], metrics[k + "_nmi"]) for k in PAPER[key[1]]))
    print("rep    mean_ACC best_ACC | paper_ACC | mean_NMI best_NMI | paper_NMI")
    for rep, (paper_acc, paper_nmi) in PAPER[key[1]].items():
        accs = [m[rep + "_acc"] for _, m in picked]
        nmis = [m[rep + "_nmi"] for _, m in picked]
        print("%-6s %8.4f %8.4f | %.4f | %8.4f %8.4f | %.4f" % (rep, statistics.mean(accs), max(accs), paper_acc, statistics.mean(nmis), max(nmis), paper_nmi))

zip_path = Path(shutil.make_archive(str(RESULTS) + "-results", "zip", RESULTS.parent, RESULTS.name))
digest = file_sha256(zip_path)
Path(str(zip_path) + ".sha256").write_text(digest + "  " + zip_path.name + chr(10), encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)